In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from concurrent.futures import ProcessPoolExecutor
from tqdm import tqdm

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/snm3C/hicluster'
donors = sorted(list(np.loadtxt(f'{projdir}/txt/donors.txt', dtype=str)))
tmp_order = ['Start', 'Sendai', 'Delayed', 'Fail1', 'Inter1',  'Inter2', 'Fail2', 'IPS']
clusts = [f'{x}_{y}' for x,y in zip(np.repeat(tmp_order, len(donors)), np.tile(donors, len(tmp_order)))]

In [ ]:
log2_min = np.log2(2500)        # ≈ 11.29
log2_max = np.log2(250000000) # ≈ 27.89
step = 0.125
log2_bins = np.arange(log2_min, log2_max + step, step)

bin_edges = 2 ** log2_bins  # convert back to basepair units
bin_edges.shape

In [ ]:
cutoff = 2e6
short_bins = bin_edges[bin_edges < cutoff][1:]
long_bins = bin_edges[bin_edges > cutoff]

In [ ]:
contact_df = pd.read_csv(f'{projdir}/csv/distance/merged.csv.gz', sep='\t', header=0, index_col=0)
contact_df.columns = bin_edges[1:]
contact_df.shape

In [ ]:
contact_df.head()

In [ ]:
indir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/mc/csv/label_transfer/snm3c_ips_subtype.csv'
clusters = pd.read_csv(indir, sep='\t', index_col=0)
clusters.shape

In [ ]:
short_sum = contact_df.loc[:,short_bins].sum(axis=1)
long_sum = contact_df.loc[:,long_bins].sum(axis=1)
sum_df = pd.DataFrame(zip(short_sum, long_sum))
sum_df.columns = ['short', 'long']
sum_df['short/long'] = sum_df['short'] / sum_df['long']
sum_df.index = contact_df.index
sum_df.shape

In [ ]:
sum_df.head()

In [ ]:
sum_df.to_csv(f'{projdir}/csv/meta/short_long.csv', sep='\t')

In [ ]:
ips_sum = sum_df.reindex(clusters.index).dropna()
ips_sum['ips_subtype'] = clusters.reindex(ips_sum.index)['ips_subtype']
ips_sum.shape

In [ ]:
ips_sum.head()

In [ ]:
tmp_main = ips_sum[ips_sum['ips_subtype'] == 'IPS_main']
cutoff = tmp_main['short/long'].median()
tmp_main_long = tmp_main[tmp_main['short/long'] < cutoff].index
tmp_main_short = tmp_main[tmp_main['short/long'] > cutoff].index

In [ ]:
tmp_alt = ips_sum[ips_sum['ips_subtype'] == 'IPS_alt']
cutoff = tmp_alt['short/long'].median()
tmp_alt_long = tmp_alt[tmp_alt['short/long'] < cutoff].index
tmp_alt_short = tmp_alt[tmp_alt['short/long'] > cutoff].index

In [ ]:
ips_sum['type'] = 'short'
ips_sum.loc[tmp_main_long,'type'] = 'long'
ips_sum.loc[tmp_alt_long,'type'] = 'long'
ips_sum['type'].value_counts()

In [ ]:
ips_sum.to_csv(f'{projdir}/csv/clusters/ips_subtypes_short_long.csv', sep='\t')

### reorder contact df

In [ ]:
cell_list = []
tmp_clusts = ['IPS_main', 'IPS_alt']
for s in tmp_clusts:
    tmp_sum = ips_sum[ips_sum['ips_subtype'] == s]
    tmp_cells = pd.DataFrame(tmp_sum.sort_values(by=['short/long']).index)
    cell_list.append(tmp_cells)
    
cell_order = pd.concat(cell_list)[0]
cell_order.shape

In [ ]:
ips_df = contact_df.reindex(cell_order)
ips_df.shape

In [ ]:
ips_df.head()

In [ ]:
ips_zscore = (ips_df.sub(ips_df.mean(axis=1), axis=0).div(ips_df.std(axis=1), axis=0)).T
ips_zscore.head()

In [ ]:
ax = sns.heatmap(ips_zscore, vmin=-2, vmax=4, cmap='coolwarm')
ax.set_xticklabels([]);

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure5'
ips_zscore.to_csv(f'{outdir}/ips_subtypes_dist_df.csv', sep='\t')

In [ ]:
ips_meta = ips_sum.reindex(cell_order)
ips_meta.shape

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure5'
ips_meta.to_csv(f'{outdir}/ips_subtypes_dist_meta.csv', sep='\t')

In [ ]:
!date